In [22]:
import pandas as pd

In [23]:
NUM_MERCHANTS = 800

P_API_KEY = 0.85
P_TEST_TXN = 0.75
P_LIVE_TXN = 0.55
P_REGULAR_USAGE = 0.35

START_DATE = pd.Timestamp("2024-01-01")
SIMULATION_DAYS = 240   # ~8 months

In [13]:
# MERCHANTS
import pandas as pd
import numpy as np
import uuid
import random
from datetime import timedelta

np.random.seed(42)

merchants = []

for _ in range(NUM_MERCHANTS):
    signup_ts = START_DATE + timedelta(days=np.random.randint(0, 60))

    merchants.append({
        "merchant_id": f"m_{uuid.uuid4().hex[:10]}",
        "signup_timestamp": signup_ts,
        "country": "NG",
        "industry": random.choice(
            ["Ecommerce", "SaaS", "Education", "Logistics", "Fintech"]
        ),
        "business_type": random.choice(["SME", "Enterprise"]),
        "signup_channel": random.choice(["Web", "Referral", "Sales"])
    })

merchants_df = pd.DataFrame(merchants)

In [14]:
# API KEYS
api_keys = []

for _, m in merchants_df.iterrows():
    if np.random.rand() < P_API_KEY:
        # TEST key
        api_keys.append({
            "api_key_id": f"k_{uuid.uuid4().hex[:10]}",
            "merchant_id": m.merchant_id,
            "created_timestamp": m.signup_timestamp + timedelta(days=1),
            "environment": "TEST",
            "key_type": "secret"
        })

        # Some create LIVE keys
        if np.random.rand() < 0.65:
            api_keys.append({
                "api_key_id": f"k_{uuid.uuid4().hex[:10]}",
                "merchant_id": m.merchant_id,
                "created_timestamp": m.signup_timestamp + timedelta(days=5),
                "environment": "LIVE",
                "key_type": "secret"
            })

api_keys_df = pd.DataFrame(api_keys)

In [ ]:
# TRANSACTIONS (same logic as python_scripts/Data_Ingestion.py)
FAILURE_RATE = 0.08               # applied to every LIVE transaction
DAILY_CHURN_PROBABILITY = 0.002   # simulated daily chance a merchant stops transacting

FAILURE_REASONS = [
    "Insufficient Funds",
    "Bank Timeout",
    "Network Error",
    "Invalid Card"
]


def create_live_transaction(merchant_id, transaction_day):
    status = "FAILED" if np.random.rand() < FAILURE_RATE else "SUCCESS"
    failure_reason = random.choice(FAILURE_REASONS) if status == "FAILED" else None

    return {
        "transaction_id": f"t_{uuid.uuid4().hex[:12]}",
        "merchant_id": merchant_id,
        "transaction_timestamp": transaction_day,
        "environment": "LIVE",
        "status": status,
        "amount": np.random.randint(2000, 80000),
        "currency": "NGN",
        "payment_method": random.choice(["card", "bank", "ussd"]),
        "failure_reason": failure_reason
    }


transactions = []

for _, m in merchants_df.iterrows():
    merchant_id = m.merchant_id

    # ---------- TEST TRANSACTIONS ----------
    if np.random.rand() < P_TEST_TXN:
        num_test = np.random.randint(5, 15)

        for i in range(num_test):
            transactions.append({
                "transaction_id": f"t_{uuid.uuid4().hex[:12]}",
                "merchant_id": merchant_id,
                "transaction_timestamp": m.signup_timestamp + timedelta(days=2 + i),
                "environment": "TEST",
                "status": "SUCCESS",
                "amount": 0.00,
                "currency": "NGN",
                "payment_method": "card",
                "failure_reason": None
            })

    # ---------- LIVE ACTIVATION ----------
    if np.random.rand() >= P_LIVE_TXN:
        continue

    activation_day = m.signup_timestamp + timedelta(days=np.random.randint(7, 20))

    activation_transaction = create_live_transaction(merchant_id, activation_day)
    transactions.append(activation_transaction)

    # Continue usage only after a successful activation transaction
    if activation_transaction["status"] == "FAILED":
        continue

    # ---------- MERCHANT USAGE PATTERN ----------
    is_power_user = np.random.rand() < P_REGULAR_USAGE

    current_day = activation_day + timedelta(days=1)
    last_active_day = activation_day + timedelta(days=SIMULATION_DAYS)

    while current_day < last_active_day:
        # Simulated merchant activity stop
        if np.random.rand() < DAILY_CHURN_PROBABILITY:
            break

        daily_txns = (
            np.random.randint(1, 4) if is_power_user
            else np.random.randint(0, 2)
        )

        for _ in range(daily_txns):
            transactions.append(create_live_transaction(merchant_id, current_day))

        current_day += timedelta(days=1)

In [ ]:
transactions_df = pd.DataFrame(transactions)

print("Merchants:", len(merchants_df))
print("API Keys:", len(api_keys_df))
print("Transactions:", len(transactions_df))

In [ ]:
# Activation rate
activated = transactions_df[
    (transactions_df.environment == "LIVE") &
    (transactions_df.status == "SUCCESS")
]["merchant_id"].nunique()

activated / merchants_df.shape[0]

In [ ]:
merchants_df.shape

In [ ]:
merchants_df.isnull().sum()

In [ ]:
api_keys_df.isnull().sum()

In [ ]:
transactions_df.isnull().sum()

In [ ]:
# INSERT INTO MARIADB (XAMPP) DATABASE: merchant_payment_intelligence
# Prerequisite: run sql_scripts/01_database_setup.sql first (creates the database and tables).

In [ ]:
# CONNECT TO MARIADB (XAMPP)
# pip install pymysql
import os
import pymysql
import pandas as pd

db_host     = os.getenv("DB_HOST", "127.0.0.1")
db_port     = int(os.getenv("DB_PORT", "3306"))
db_user     = os.getenv("DB_USER", "root")
db_password = os.getenv("DB_PASSWORD", "")   # XAMPP root has an empty password by default
database    = os.getenv("DB_NAME", "merchant_payment_intelligence")

conn = pymysql.connect(
    host=db_host,
    port=db_port,
    user=db_user,
    password=db_password,
    database=database,
    charset="utf8mb4",
    autocommit=False
)

cursor = conn.cursor()


def to_python(value):
    """Convert pandas/numpy values to types PyMySQL can send (NaN -> NULL)."""
    if pd.isna(value):
        return None
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    if hasattr(value, "item"):      # numpy scalar -> plain Python scalar
        return value.item()
    return value


def df_to_rows(df, columns):
    return [
        tuple(to_python(v) for v in row)
        for row in df[columns].itertuples(index=False, name=None)
    ]

In [ ]:
insert_query = """
INSERT INTO merchants (
    merchant_id,
    signup_timestamp,
    country,
    industry,
    business_type,
    signup_channel
)
VALUES (%s, %s, %s, %s, %s, %s)
"""

merchant_columns = [
    "merchant_id", "signup_timestamp", "country",
    "industry", "business_type", "signup_channel"
]

merchant_rows = df_to_rows(merchants_df, merchant_columns)

cursor.executemany(insert_query, merchant_rows)
conn.commit()

print(f"Inserted {len(merchant_rows)} merchants")

In [ ]:
insert_api_keys = """
INSERT INTO api_keys (
    api_key_id,
    merchant_id,
    created_timestamp,
    environment,
    key_type
)
VALUES (%s, %s, %s, %s, %s)
"""

api_key_columns = [
    "api_key_id", "merchant_id", "created_timestamp",
    "environment", "key_type"
]

api_key_rows = df_to_rows(api_keys_df, api_key_columns)

cursor.executemany(insert_api_keys, api_key_rows)
conn.commit()

print(f"Inserted {len(api_key_rows)} API keys")

In [ ]:
insert_transactions = """
INSERT INTO transactions (
    transaction_id,
    merchant_id,
    transaction_timestamp,
    environment,
    status,
    amount,
    currency,
    payment_method,
    failure_reason
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

transaction_columns = [
    "transaction_id", "merchant_id", "transaction_timestamp", "environment",
    "status", "amount", "currency", "payment_method", "failure_reason"
]

transaction_rows = df_to_rows(transactions_df, transaction_columns)

batch_size = 1000

for start in range(0, len(transaction_rows), batch_size):
    cursor.executemany(insert_transactions, transaction_rows[start:start + batch_size])
    conn.commit()

print(f"Inserted {len(transaction_rows):,} transactions successfully")

cursor.close()
conn.close()

In [1]:
!jupyter nbconvert --to python Data_Ingestion.ipynb

[NbConvertApp] Converting notebook Data_Ingestion.ipynb to python
[NbConvertApp] Writing 7772 bytes to Data_Ingestion.py
